# Task 3: Predictive Customer Churn Classifier (Random Forest & XGBoost)
**Goal:** Predict telecom customer churn, handle class imbalance with SMOTE, train Random Forest and XGBoost, and evaluate with ROC-AUC (target > 0.80).

**Data:** place the Telco Customer Churn CSV at `data/telco_churn.csv`. Adjust `DATA_PATH` if your file has a different name.

## 1. Imports and data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

sns.set_style("whitegrid")
RANDOM_STATE = 42
DATA_PATH = Path("../data/telco_churn.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
df.info()

## 2. Clean the data
`TotalCharges` is stored as text and contains blank strings. Convert it to numeric and handle the resulting nulls.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Null TotalCharges after conversion:", df["TotalCharges"].isna().sum())
# Blank TotalCharges belong to brand-new customers (tenure = 0), so 0 is the logical value.
df["TotalCharges"] = df["TotalCharges"].fillna(0)

df["Churn"] = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["customerID"])
print("Remaining nulls:", int(df.isna().sum().sum()))
print()
print("Churn rate:", round(df["Churn"].mean(), 3))
df["Churn"].value_counts().rename({0: "No churn", 1: "Churn"}).plot(kind="bar", rot=0, title="Class distribution (imbalanced)")
plt.show()

## 3. Train/test split and preprocessing
Stratified 80/20 split. Categorical features are one-hot encoded; numeric features pass through (tree models do not need scaling).

In [ ]:
X = df.drop(columns="Churn")
y = df["Churn"]
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

def make_pre():
    return ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)],
        remainder="passthrough", verbose_feature_names_out=False)

## 4. Models with SMOTE
SMOTE sits inside an `imblearn` Pipeline, so oversampling is applied **only to the training folds** during cross-validation and never leaks synthetic samples into validation or test data.

In [ ]:
models = {
    "Random Forest": ImbPipeline([
        ("pre", make_pre()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                       random_state=RANDOM_STATE, n_jobs=-1))]),
    "XGBoost": ImbPipeline([
        ("pre", make_pre()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
}

## 5. Cross-validated ROC-AUC (5-fold stratified, training data)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:14s} CV ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 6. Fit on the full training set and evaluate on the test set

In [ ]:
fitted, proba, results = {}, {}, []
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba[name] = model.predict_proba(X_test)[:, 1]
    results.append({"Model": name, "Test ROC-AUC": roc_auc_score(y_test, proba[name])})

results_df = pd.DataFrame(results).set_index("Model").round(4)
display(results_df)
print("Target ROC-AUC > 0.80 met:", bool((results_df["Test ROC-AUC"] > 0.80).any()))

for name, model in fitted.items():
    print("=== " + name + " ===")
    print(classification_report(y_test, model.predict(X_test), target_names=["No churn", "Churn"]))

## 7. ROC curves

In [ ]:
plt.figure(figsize=(7, 6))
for name, p in proba.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves on the test set")
plt.legend(loc="lower right")
plt.show()

## 8. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, model) in zip(axes, fitted.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["No churn", "Churn"], yticklabels=["No churn", "Churn"], ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 9. Top 10 most influential features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, model) in zip(axes, fitted.items()):
    feat_names = model.named_steps["pre"].get_feature_names_out()
    imp = pd.Series(model.named_steps["clf"].feature_importances_, index=feat_names)
    imp.nlargest(10).sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"{name}: top 10 feature importances")
plt.tight_layout()
plt.show()

## 10. Conclusions
- Churn is imbalanced, so accuracy alone is misleading; ROC-AUC and recall on the churn class are more informative.
- SMOTE inside the pipeline balances the classes during training without contaminating validation or test data.
- Contract type, tenure and monthly charges are typically among the strongest churn drivers on this dataset. Check the importance charts above for your run.
- Compare the test ROC-AUC values in section 6 against the 0.80 target and note which model you would deploy.